<a href="https://colab.research.google.com/github/padminidokka/AI-Based-Dental-Radiograph-Analysis-System/blob/main/OpenCV.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install opencv-python

In [ ]:
import os
import cv2
import numpy as np

IMAGE_FOLDER = "/content/drive/MyDrive/Dental/opg_images"
LABEL_FOLDER = "/content/DentalDataset/labels/train"

os.makedirs(LABEL_FOLDER, exist_ok=True)

In [ ]:
def detect_teeth(image):

    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

    # improve contrast
    gray = cv2.equalizeHist(gray)

    # blur
    blur = cv2.GaussianBlur(gray,(5,5),0)

    # threshold
    _, thresh = cv2.threshold(blur,120,255,cv2.THRESH_BINARY)

    # remove noise
    kernel = np.ones((3,3),np.uint8)
    thresh = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, kernel)

    return thresh

In [ ]:
def contours_to_yolo(contours, img_w, img_h):

    yolo_boxes = []

    for cnt in contours:

        x,y,w,h = cv2.boundingRect(cnt)

        if w*h < 500:   # remove tiny noise
            continue

        # convert to YOLO format
        x_center = (x + w/2) / img_w
        y_center = (y + h/2) / img_h
        width = w / img_w
        height = h / img_h

        yolo_boxes.append([0,x_center,y_center,width,height])  # class 0 = tooth

    return yolo_boxes

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
images = os.listdir(IMAGE_FOLDER)

for img_name in images:

    img_path = os.path.join(IMAGE_FOLDER,img_name)

    image = cv2.imread(img_path)

    if image is None:
        continue

    h,w,_ = image.shape

    mask = detect_teeth(image)

    contours,_ = cv2.findContours(mask,cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_SIMPLE)

    boxes = contours_to_yolo(contours,w,h)

    label_name = img_name.split(".")[0] + ".txt"
    label_path = os.path.join(LABEL_FOLDER,label_name)

    with open(label_path,"w") as f:

        for box in boxes:

            f.write(" ".join(map(str,box))+"\n")

In [ ]:
def visualize_boxes(image_path,label_path):

    img = cv2.imread(image_path)
    h,w,_ = img.shape

    with open(label_path) as f:
        lines = f.readlines()

    for line in lines:

        cls,x,y,ww,hh = map(float,line.split())

        x1 = int((x-ww/2)*w)
        y1 = int((y-hh/2)*h)
        x2 = int((x+ww/2)*w)
        y2 = int((y+hh/2)*h)

        cv2.rectangle(img,(x1,y1),(x2,y2),(0,255,0),2)

    cv2.imshow("boxes",img)
    cv2.waitKey(0)

In [ ]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 31.6 MB/s eta 0:00:00


In [ ]:
import os

print(os.listdir("/content"))

['.config', 'runs', 'yolov8n.pt', 'DentalDataset', 'drive', 'sample_data']


In [ ]:
print(os.listdir("/content/DentalDataset"))

['labels']


In [ ]:
import os

print("labels/train:", os.listdir("/content/DentalDataset/labels/train")[:5])

labels/train: ['135.txt', '273.txt', '248.txt', '130.txt', '50.txt']


In [ ]:
import os
import shutil

source = "/content/drive/MyDrive/Dental/opg_images"
dest = "/content/DentalDataset/images/train"

os.makedirs(dest, exist_ok=True)

for img in os.listdir(source):

    if img.endswith((".jpg",".png",".jpeg")):
        shutil.copy(
            os.path.join(source,img),
            os.path.join(dest,img)
        )

print("Images copied")

Images copied


In [ ]:
!tree /content/DentalDataset

/bin/bash: line 1: tree: command not found


In [ ]:
!head /content/DentalDataset/labels/train/*.txt

Streaming output truncated to the last 5000 lines.

==> /content/DentalDataset/labels/train/129.txt <==
0 0.9784419402253797 0.84375 0.016658500734933857 0.05859375
0 0.021068103870651642 0.84375 0.016658500734933857 0.05859375
0 0.5 0.5 1.0 1.0

==> /content/DentalDataset/labels/train/12.txt <==
0 0.9784419402253797 0.84375 0.016658500734933857 0.05859375
0 0.021068103870651642 0.84375 0.016658500734933857 0.05859375
0 0.5 0.5 1.0 1.0

==> /content/DentalDataset/labels/train/130.txt <==
0 0.4970602645761881 0.99462890625 0.03184713375796178 0.0107421875
0 0.23958843704066635 0.8330078125 0.014698677119059285 0.01953125
0 0.9784419402253797 0.84375 0.016658500734933857 0.05859375
0 0.021068103870651642 0.84375 0.016658500734933857 0.05859375
0 0.5 0.4990234375 1.0 0.998046875

==> /content/DentalDataset/labels/train/131.txt <==
0 0.9784419402253797 0.84375 0.016658500734933857 0.05859375
0 0.021068103870651642 0.84375 0.016658500734933857 0.05859375
0 0.9321411073003429 0.484375 0.0132

In [ ]:
!ls /content

DentalDataset  drive  runs  sample_data  yolov8n.pt


In [ ]:
yaml_content = """
path: /content/DentalDataset

train: images/train
val: images/val

names:
  0: tooth
"""

with open("/content/dental.yaml", "w") as f:
    f.write(yaml_content)

print("dental.yaml created successfully")

dental.yaml created successfully


In [ ]:
!ls /content

DentalDataset  dental.yaml  drive  runs  sample_data  yolov8n.pt


In [ ]:
import os

os.makedirs("/content/DentalDataset/images/val", exist_ok=True)
os.makedirs("/content/DentalDataset/labels/val", exist_ok=True)

print("Validation folders created")

Validation folders created


In [ ]:
import os
import random
import shutil

train_img = "/content/DentalDataset/images/train"
val_img = "/content/DentalDataset/images/val"

train_lbl = "/content/DentalDataset/labels/train"
val_lbl = "/content/DentalDataset/labels/val"

images = os.listdir(train_img)

# move 15% of images to validation
val_count = int(len(images) * 0.15)

val_images = random.sample(images, val_count)

for img in val_images:

    shutil.move(
        os.path.join(train_img, img),
        os.path.join(val_img, img)
    )

    label = img.replace(".jpg",".txt")

    if os.path.exists(os.path.join(train_lbl,label)):
        shutil.move(
            os.path.join(train_lbl,label),
            os.path.join(val_lbl,label)
        )

print("Validation split created")

Validation split created


In [ ]:
!apt-get install tree

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  tree
0 upgraded, 1 newly installed, 0 to remove and 2 not upgraded.
Need to get 47.9 kB of archives.
After this operation, 116 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/universe amd64 tree amd64 2.0.2-1 [47.9 kB]
Fetched 47.9 kB in 0s (159 kB/s)
Selecting previously unselected package tree.
(Reading database ... 117540 files and directories currently installed.)
Preparing to unpack .../tree_2.0.2-1_amd64.deb ...
Unpacking tree (2.0.2-1) ...
Setting up tree (2.0.2-1) ...
Processing triggers for man-db (2.10.2-1) ...


In [ ]:
!tree /content/DentalDataset

/content/DentalDataset
├── images
│   ├── train
│   │   ├── 100.jpg
│   │   ├── 101.jpg
│   │   ├── 102.jpg
│   │   ├── 103.jpg
│   │   ├── 104.jpg
│   │   ├── 105.jpg
│   │   ├── 106.jpg
│   │   ├── 107.jpg
│   │   ├── 108.jpg
│   │   ├── 109.jpg
│   │   ├── 10.jpg
│   │   ├── 110.jpg
│   │   ├── 111.jpg
│   │   ├── 112.jpg
│   │   ├── 113.jpg
│   │   ├── 114.jpg
│   │   ├── 116.jpg
│   │   ├── 117.jpg
│   │   ├── 118.jpg
│   │   ├── 119.jpg
│   │   ├── 11.jpg
│   │   ├── 120.jpg
│   │   ├── 121.jpg
│   │   ├── 122.jpg
│   │   ├── 123.jpg
│   │   ├── 125.jpg
│   │   ├── 126.jpg
│   │   ├── 127.jpg
│   │   ├── 129.jpg
│   │   ├── 12.jpg
│   │   ├── 130.jpg
│   │   ├── 131.jpg
│   │   ├── 132.jpg
│   │   ├── 133.jpg
│   │   ├── 134.jpg
│   │   ├── 135.jpg
│   │   ├── 136.jpg
│   │   ├── 137.jpg
│   │   ├── 138.jpg
│   │   ├── 139.jpg
│   │   ├── 13.jpg
│   │   ├── 140.jpg
│   │   ├── 141.jpg
│   │   ├── 142.jpg
│   │   ├── 143.jpg
│   │   ├── 144.jpg
│   │   ├── 145.jpg
│   │   ├── 147.

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")

model.train(
    data="/content/dental.yaml",
    epochs=5,
    imgsz=640
)

Ultralytics 8.4.21 🚀 Python-3.12.12 torch-2.10.0+cpu CPU (Intel Xeon CPU @ 2.20GHz)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/dental.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=5, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train4, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspective=0.0, plo

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7bf61b8ebad0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.048048, 

In [ ]:
os.listdir("/content/DentalDataset/images/train")[:5]
os.listdir("/content/DentalDataset/labels/train")[:5]

['135.txt', '273.txt', '130.txt', '288.txt', '95.txt']

In [ ]:
from ultralytics import YOLO

model = YOLO("runs/detect/train/weights/best.pt")

results = model("/content/opg.jpg")

In [ ]:
boxes = results[0].boxes.xyxy.cpu().numpy()

# boxes = [x1,y1,x2,y2]

In [ ]:
centers = []

for box in boxes:
    x1,y1,x2,y2 = box
    cx = (x1+x2)/2
    cy = (y1+y2)/2
    centers.append((cx,cy,box))

In [ ]:
import numpy as np

centers = np.array(centers,dtype=object)

midline = np.mean(centers[:,1])

upper_teeth = [c for c in centers if c[1] < midline]
lower_teeth = [c for c in centers if c[1] >= midline]

In [ ]:
upper_teeth = sorted(upper_teeth,key=lambda x:x[0])
lower_teeth = sorted(lower_teeth,key=lambda x:x[0])

In [ ]:
upper_fdi = [18,17,16,15,14,13,12,11,21,22,23,24,25,26,27,28]
lower_fdi = [48,47,46,45,44,43,42,41,31,32,33,34,35,36,37,38]

fdi_labels = {}

for i,tooth in enumerate(upper_teeth[:16]):
    fdi_labels[tuple(tooth[2])] = upper_fdi[i]

for i,tooth in enumerate(lower_teeth[:16]):
    fdi_labels[tuple(tooth[2])] = lower_fdi[i]

In [ ]:
import cv2

img = cv2.imread("/content/opg.jpg")

for box,number in fdi_labels.items():

    x1,y1,x2,y2 = map(int,box)

    cv2.rectangle(img,(x1,y1),(x2,y2),(0,255,0),2)

    cv2.putText(img,str(number),
                (x1,y1-5),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.6,
                (255,0,0),
                2)

cv2.imshow("FDI Teeth",img)
cv2.waitKey(0)